# Notes Data Warehouse & BigQuery - Module 3 DE Zoomcamp

## PARTIE 1 : DATA WAREHOUSE - CONCEPTS FONDAMENTAUX

### Qu'est-ce qu'un Data Warehouse ?

**Data Warehouse (DWH)** = Système de stockage centralisé optimisé pour l'**analyse** de grandes quantités de données structurées.

**Data Warehouse vs Base de données transactionnelle (OLTP) :**
- **OLTP** (PostgreSQL, MySQL) : Optimisé pour les transactions (INSERT, UPDATE, DELETE), normalisé, lecture ligne par ligne
- **OLAP** (BigQuery, Snowflake) : Optimisé pour l'analyse (SELECT, GROUP BY, COUNT), dénormalisé, lecture colonne par colonne

**Ce qu'un Data Warehouse N'EST PAS :**
- Pas une base de données transactionnelle (pas fait pour du INSERT/UPDATE fréquent)
- Pas un Data Lake (DWH = données structurées et transformées)
- Pas un Data Mart (Data Mart = sous-ensemble d'un DWH pour un département)

---

### Pourquoi c'est important pour le Data Engineer ?

**Lien avec les équipes métier = clé du succès !**

Un Data Engineer ne peut pas optimiser dans son coin. Il doit **discuter avec les équipes** pour comprendre :
- Quelles colonnes l'équipe Analytics filtre le plus souvent ?
- Sur quelles périodes l'équipe BI fait ses rapports ?
- Quelles agrégations l'équipe Data Science utilise quotidiennement ?

**Sans discussion** → Optimisation à l'aveugle → Gaspillage d'argent

**Avec discussion** → Bonne stratégie de partition/clustering → Économies massives

```
Exemple concret :
Sans parler aux équipes   → Table mal optimisée : 310 MB scan par query
Avec les besoins des équipes → Table optimisée : 25 MB scan par query
= 12x moins de coûts sur des millions de queries 💰
```

---

### Architecture Data Warehouse

**Le pipeline typique :**

```
Sources (PostgreSQL, APIs, CSV)
    ↓
Data Lake (GCS Bucket) ← Fichiers Parquet/CSV
    ↓
Data Warehouse (BigQuery) ← Tables optimisées
    ↓
BI Tools (Looker, Tableau, Data Studio)
```

**BigQuery = Data Warehouse de Google Cloud Platform (GCP)**
- Serverless (pas de serveur à gérer)
- Stockage columnar (par colonne, pas par ligne)
- Séparation stockage / compute (tu paies seulement pour le scan)

---

## PARTIE 2 : TYPES DE TABLES DANS BIGQUERY

### External Table

**Définition :** Pointeur vers des fichiers dans GCS. Les données restent dans le bucket.

```sql
CREATE OR REPLACE EXTERNAL TABLE zoomcamp.external_yellow_tripdata
OPTIONS(
  format = 'PARQUET',
  uris = ['gs://mon-bucket/yellow_tripdata_2024-0*.parquet']
);
```

**Caractéristiques :**
- ❌ Données restent dans GCS (pas dans BigQuery)
- ❌ Pas de partitioning possible
- ❌ Pas de clustering possible
- ❌ Estimation de coût imprécise (affiche souvent "0 B" même si ça scanne !)
- ✅ Pas de coût de storage BigQuery
- ✅ Utile pour explorer des données temporaires

**⚠️ PIÈGE CLASSIQUE :**

```
Estimation affichée AVANT exécution : "0 B"  ← TROMPEUR
Consommation réelle APRÈS exécution : 310 MB  ← VÉRITÉ

Toujours vérifier les Job Details après exécution pour les External Tables !
```

---

### Regular Table (Native Table)

**Définition :** Données copiées et stockées dans BigQuery.

```sql
-- Créer depuis une External Table
CREATE OR REPLACE TABLE zoomcamp.yellow_tripdata AS
SELECT * FROM zoomcamp.external_yellow_tripdata;
```

**Caractéristiques :**
- ✅ Données dans BigQuery storage
- ✅ Estimation de coût PRÉCISE avant exécution
- ✅ Partitioning possible 🔥
- ✅ Clustering possible 🔥
- ✅ Toutes les optimisations BigQuery disponibles
- ⚠️ Coût de storage (~$0.02/GB/mois)

---

### Materialized View

**Définition :** Vue dont le résultat est pré-calculé et stocké. Se refresh automatiquement.

```sql
CREATE MATERIALIZED VIEW zoomcamp.daily_revenue AS
SELECT 
  DATE(order_date) as date,
  SUM(total_amount) as revenue,
  COUNT(*) as order_count
FROM orders
GROUP BY date;
```

**Différence avec Regular Table :**

| | Regular Table | Materialized View |
|---|---|---|
| **Données** | Statiques (tu gères) | Auto-refresh automatique |
| **Contenu** | N'importe quoi | Résultat d'une query |
| **Usage** | Table principale | Agrégations fréquentes pré-calculées |
| **Limites** | Aucune | Pas de JOINs, agrégations limitées |

---

### Comparaison des Types de Tables

```
Query de comparaison sur même données :

External Table  → 310 MB scan, durée 1 sec
Regular Table   → 310 MB scan, durée 1 sec  
Partitioned     → 25 MB scan, durée 0 sec   ← 12x moins ! 🚀
```

---

## PARTIE 3 : PARTITIONING

### Concept fondamental

**Partitioning** = Découper une table en segments physiques séparés selon une colonne.

Analogie : Classer tes documents dans des classeurs par année/mois. Tu cherches doc de 2024 ? Tu ouvres seulement le classeur 2024, pas tous les autres.

### Types de Partitioning

**1. Time-unit column partitioning** (le plus courant)

```sql
CREATE OR REPLACE TABLE zoomcamp.trips_partitioned
PARTITION BY DATE(tpep_pickup_datetime)  -- Granularité : DAY par défaut
AS SELECT * FROM zoomcamp.external_trips;
```

Granularités disponibles : HOUR, DAY, MONTH, YEAR

**2. Ingestion time partitioning** (_PARTITIONTIME)

```sql
CREATE OR REPLACE TABLE zoomcamp.trips_by_ingestion
PARTITION BY DATE(_PARTITIONTIME)
AS SELECT * FROM zoomcamp.external_trips;
```

Utile si pas de colonne temporelle naturelle.

**3. Integer range partitioning**

```sql
CREATE OR REPLACE TABLE zoomcamp.trips_by_vendor
PARTITION BY RANGE_BUCKET(VendorID, GENERATE_ARRAY(1, 100, 5))
AS SELECT * FROM zoomcamp.external_trips;
```

### Avantages du Partitioning

**Partition Pruning** = BigQuery scanne SEULEMENT les partitions nécessaires

```sql
-- Sans partition : scan 1 an = 100 GB 💸
SELECT COUNT(*) FROM trips;

-- Avec partition : scan 1 jour = 274 MB ✅
SELECT COUNT(*) FROM trips_partitioned
WHERE DATE(tpep_pickup_datetime) = '2024-01-15';
```

**Coût prévisible AVANT l'exécution** :
- BigQuery affiche exactement combien de MB sera scanné
- Tu peux budgéter précisément tes coûts

**Gestion du cycle de vie :**
```sql
CREATE TABLE trips
PARTITION BY DATE(pickup_datetime)
OPTIONS(
  partition_expiration_days=90,       -- Auto-suppression après 90 jours
  require_partition_filter=true       -- Force le WHERE sur partition (sécurité !)
);
```

### Limites du Partitioning

- Maximum **4000 partitions** par table
- Partitioning trop granulaire (par heure sur des années) = dépassement facile
- Ne fonctionne pas si pas de colonne temporelle ou numérique adaptée

---

## PARTIE 4 : CLUSTERING

### Concept fondamental

**Clustering** = Organiser les données à l'intérieur des partitions (ou de la table) selon l'ordre de certaines colonnes.

Analogie : Dans ta bibliothèque (partition), tu ranges les livres par auteur (cluster). Tu cherches Victor Hugo ? Tu vas directement à la section "H", pas toute la bibliothèque.

### Comment ça marche

BigQuery stocke les données dans des **"blocks"** :
1. Les données avec valeurs similaires dans les colonnes de clustering → mêmes blocks
2. Lors d'une query avec filtre sur ces colonnes → BigQuery **skip** les blocks non pertinents

```sql
-- Table partitionnée ET clusterisée (combo optimal)
CREATE OR REPLACE TABLE zoomcamp.trips_optimized
PARTITION BY DATE(tpep_pickup_datetime)
CLUSTER BY VendorID, payment_type
AS SELECT * FROM zoomcamp.external_trips;
```

### ⚠️ DIFFÉRENCE CLÉE : Coût Prévisible

**Avec PARTITIONING :**
```
BigQuery dit AVANT : "Je vais scanner 25 MB"
Après exécution : 25 MB  ← PRÉCIS ✅
```

**Avec CLUSTERING :**
```
BigQuery dit AVANT : "Je vais scanner 0 B to 2 GB"  ← IMPRÉCIS ⚠️
Après exécution : 25 MB  ← Le vrai chiffre apparaît seulement APRÈS
```

Pourquoi ? Partitioning = découpage **statique** connu à l'avance. Clustering = organisation **dynamique** évaluée pendant l'exécution.

### Ordre des Colonnes de Clustering

**RÈGLE : Haute cardinalité → Basse cardinalité**

**Cardinalité** = Nombre de valeurs distinctes dans une colonne
- Haute cardinalité : user_id (10 millions de valeurs) → Haute sélectivité
- Basse cardinalité : payment_type (5 valeurs) → Faible sélectivité

```sql
-- ✅ BON : Haute → Basse cardinalité
CLUSTER BY user_id, country, payment_type
-- user_id concentre les données dans peu de blocks → skip max

-- ❌ MAUVAIS : Basse → Haute cardinalité  
CLUSTER BY payment_type, country, user_id
-- payment_type (5 valeurs) = grandes sections encore mélangées
```

**Pourquoi ?** Haute cardinalité en premier = segmentation maximale dès le départ = données concentrées dans moins de blocks = scan minimal.

**Exception :** Si tes queries filtrent TOUJOURS sur une colonne basse cardinalité en premier → mets-la en premier.

### Cluster Sans Partition

**OUI, c'est possible et courant !**

```sql
-- Cluster seul (sans partition)
CREATE TABLE zoomcamp.orders_clustered
CLUSTER BY category, brand, status
AS SELECT * FROM source;
```

Quand utiliser ?
- Pas de colonne temporelle naturelle
- Dataset trop petit pour le partitioning
- Queries filtrent sur colonnes catégorielles

### Limites du Clustering

- Max **4 colonnes** de clustering
- Coût non prévisible avant exécution
- Le clustering aide pour **WHERE** et **GROUP BY** sur les colonnes clusterisées
- Le clustering N'AIDE PAS directement pour un **ORDER BY** seul

---

## PARTIE 5 : STRATÉGIE PARTITION + CLUSTERING

### Quand utiliser quoi ?

**PARTITION seulement quand :**
- Tu as une colonne temporelle utilisée dans les WHERE
- Tu veux un contrôle précis des coûts AVANT exécution
- Tu veux gérer le cycle de vie des données (expiration)

**CLUSTERING seulement quand :**
- Pas de colonne temporelle
- Queries filtrent sur colonnes catégorielles
- Tu as déjà les 4000 partitions max

**LES DEUX quand :**
- Tu veux le meilleur des deux mondes !
- Pattern classique : Partition par date + Cluster par colonnes fréquemment filtrées

### Exemple : Quelle stratégie pour cette query ?

```
Query : Filtre TOUJOURS sur tpep_dropoff_datetime + ORDER BY VendorID

Options :
A) PARTITION BY tpep_dropoff_datetime + CLUSTER BY VendorID  ✅
B) CLUSTER BY tpep_dropoff_datetime + CLUSTER BY VendorID   ❌ (Impossible, 1 seul CLUSTER BY)
C) CLUSTER BY tpep_dropoff_datetime + PARTITION BY VendorID ❌ (VendorID = basse cardinalité)
D) PARTITION BY tpep_dropoff_datetime + PARTITION BY VendorID ❌ (Impossible, 1 seule PARTITION)
```

**Réponse A** : Partition sur la colonne filtrée + Cluster sur VendorID même si c'est juste pour ORDER BY (seule option techniquement valide)

Note : Le clustering sur VendorID n'aide pas directement l'ORDER BY mais n'a pas de coût et peut aider si des WHERE sur VendorID sont ajoutés plus tard.

### Résultats Réels Observés

```
Même query sur données Yellow Taxi 2024 :

Table normale (materialized_yellow_tripdata) :
→ Bytes processed : 310.24 MB
→ Duration : 1 sec

Table partitionnée + clusterisée :
→ Bytes processed : 25.05 MB
→ Duration : 0 sec

= 12x moins de scan ! 💰
```

---

## PARTIE 6 : BEST PRACTICES BIGQUERY

### 🔥 RÈGLE #1 : JAMAIS DE SELECT *

BigQuery = stockage columnar = tu paies pour chaque colonne scannée.

```sql
-- ❌ CATASTROPHIQUE
SELECT * FROM trips;  -- Scan 100 GB (20 colonnes)

-- ✅ OPTIMAL
SELECT trip_id, pickup_datetime, total_amount FROM trips;  -- Scan 15 GB (3 colonnes)
```

**Exception : COUNT(*)**
```sql
-- ✅ GRATUIT (BigQuery lit les métadonnées, pas les données)
SELECT COUNT(*) FROM trips;  -- 0 bytes !

-- 💸 COÛTEUX (doit scanner la colonne pour compter les non-NULL)
SELECT COUNT(trip_id) FROM trips;  -- 155 MB !
```

### 🔥 RÈGLE #2 : FILTRER TÔT ET FORT

```sql
-- ❌ Filtre après scan
SELECT vendor_id, AVG(total_amount)
FROM trips
GROUP BY vendor_id
HAVING AVG(total_amount) > 50;  -- Groupe tout puis filtre

-- ✅ Filtre avant scan
SELECT vendor_id, AVG(total_amount)
FROM trips
WHERE total_amount > 10  -- Réduit les données AVANT l'agrégation
GROUP BY vendor_id
HAVING AVG(total_amount) > 50;
```

### 🔥 RÈGLE #3 : ÉVITER LES FONCTIONS DANS LE WHERE SUR COLONNES PARTITIONNÉES

```sql
-- ❌ MAUVAIS : La fonction désactive le partition pruning !
SELECT * FROM trips
WHERE EXTRACT(YEAR FROM pickup_datetime) = 2024;
-- Scan : TOUTES les partitions 💀

-- ✅ BON : Filtre direct, partition pruning actif
SELECT * FROM trips
WHERE pickup_datetime BETWEEN '2024-01-01' AND '2024-12-31';
-- Scan : seulement 2024 🚀
```

### 🔥 RÈGLE #4 : REQUIRE_PARTITION_FILTER

```sql
CREATE TABLE trips
PARTITION BY DATE(pickup_datetime)
OPTIONS(require_partition_filter=true);

-- Sans filtre partition → ERREUR (sécurité contre les scans complets accidentels)
SELECT COUNT(*) FROM trips;
-- ❌ Error: Cannot query without partition filter

-- Avec filtre → OK
SELECT COUNT(*) FROM trips
WHERE DATE(pickup_datetime) = '2024-01-15';
-- ✅
```

### ⚠️ AUTRES BEST PRACTICES IMPORTANTES

**JOINs : Table large à gauche, petite à droite**
```sql
-- ✅ BON (BigQuery broadcast la petite table à droite)
SELECT * FROM large_table l
JOIN small_table s ON l.id = s.id;
```

**Préférer les Window Functions aux Self-JOINs**
```sql
-- ❌ LENT : Self-join
-- ✅ RAPIDE
SELECT 
  trip_id,
  LAG(pickup_datetime) OVER (
    PARTITION BY driver_id ORDER BY pickup_datetime
  ) as previous_trip
FROM trips;
```

**Approximations pour les grands volumes**
```sql
-- ❌ Exact mais lent sur 1 milliard de rows
SELECT COUNT(DISTINCT user_id) FROM events;

-- ✅ Approximatif (±1-2%) mais 10x plus rapide
SELECT APPROX_COUNT_DISTINCT(user_id) FROM events;
```

**CTEs pour la lisibilité**
```sql
-- ✅ CTEs = lisible, maintenable, même performance
WITH filtered_orders AS (
  SELECT * FROM orders WHERE order_date >= '2024-01-01'
),
user_counts AS (
  SELECT user_id, COUNT(*) as order_count FROM filtered_orders GROUP BY user_id
)
SELECT * FROM user_counts WHERE order_count > 5;
```

---

## PARTIE 7 : DÉNORMALISATION

### Normalisation vs Dénormalisation

**NORMALISATION** = Plusieurs tables liées par des IDs (principe des DB relationnelles)

```
Table orders : order_id, customer_id, product_id, amount
Table customers : customer_id, name, email, country
Table products : product_id, name, category
```

✅ Avantages : Pas de duplication, cohérence, moins d'espace
❌ Inconvénient : Nécessite des JOINs = shuffle de données = lent dans BigQuery

**DÉNORMALISATION** = Tout dans 1 table (ou peu de tables)

```
Table orders_denormalized :
order_id, customer_id, customer_name, customer_email, customer_country,
product_id, product_name, product_category, amount
```

✅ Avantages : Pas de JOINs = pas de shuffle = très rapide !
❌ Inconvénient : Duplication des données (customer_name répété 1000 fois)

### Pourquoi BigQuery préfère la dénormalisation

BigQuery = architecture distribuée. Les JOINs nécessitent du **shuffle** (déplacement de données entre workers) = lent et coûteux.

```
Table normalisée (3 tables, 2 JOINs) :
→ Scan : 50 GB + shuffle overhead
→ Temps : 15 secondes

Table dénormalisée (1 table, 0 JOIN) :
→ Scan : 15 GB (seulement les colonnes nécessaires)
→ Temps : 2 secondes
```

### Architecture Hybride Recommandée

```
1. SOURCE : Base de données normalisée (PostgreSQL)
      ↓
2. ETL : Pipeline qui dénormalise (Kestra, Airflow)
      ↓
3. BIGQUERY : Table dénormalisée (prête pour l'analyse)
      ↓
4. ANALYTICS : Queries rapides, pas de JOINs
```

### Quand Dénormaliser / Garder Normalisé

| Situation | Stratégie |
|-----------|-----------|
| Données rarement modifiées | ✅ Dénormaliser |
| Queries fréquentes avec mêmes JOINs | ✅ Dénormaliser |
| Ratio lecture/écriture élevé | ✅ Dénormaliser |
| Données changeant fréquemment | ❌ Garder normalisé |
| Tables de référence très grandes (>10 GB) | ❌ Garder normalisé |
| Besoin de cohérence stricte | ❌ Garder normalisé |

---

## MONITORING DES COÛTS

### Vérifier la consommation réelle

```sql
-- Queries les plus coûteuses des 7 derniers jours
SELECT
  user_email,
  query,
  total_bytes_processed / POW(10,9) as gb_processed,
  total_slot_ms / 1000 as slot_seconds,
  creation_time
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
ORDER BY total_bytes_processed DESC
LIMIT 100;
```

### Voir les partitions d'une table

```sql
SELECT 
  table_name,
  partition_id,
  total_rows,
  total_logical_bytes / POW(10,9) as size_gb
FROM `project.dataset.INFORMATION_SCHEMA.PARTITIONS`
WHERE table_name = 'trips_partitioned'
ORDER BY partition_id DESC;
```

---

## Récapitulatif Final

### Data Warehouse - Points clés

- **OLTP vs OLAP** : Transactionnel vs Analytique (deux usages différents)
- **BigQuery = columnar** : Tu paies par colonne scannée (jamais SELECT *)
- **Communication équipes** : Comprendre les besoins = optimiser en conséquence = économies

### Types de Tables - Points clés

- **External Table** : Données dans GCS, estimation "0 B" trompeuse, pas d'optimisation possible
- **Regular Table** : Données dans BigQuery, estimation précise, optimisations disponibles
- **Materialized View** : Résultat pré-calculé et auto-refreshé, pour agrégations fréquentes

### Partitioning & Clustering - Points clés

- **Partition** : Découpage physique par date → coût prévisible AVANT exécution
- **Cluster** : Organisation par colonnes catégorielles → coût connu APRÈS exécution
- **Ordre clustering** : Haute cardinalité → Basse cardinalité (sélectivité maximale)
- **Combo** : PARTITION BY date + CLUSTER BY colonnes filtrées = résultats optimaux
- **JAMAIS de fonction** dans WHERE sur colonnes partitionnées (désactive le pruning !)
- **require_partition_filter** = sécurité contre les scans complets accidentels

### Best Practices - Points clés

- ✅ SELECT colonnes spécifiques (jamais SELECT *)
- ✅ COUNT(*) gratuit (métadonnées), COUNT(col) payant (scan)
- ✅ WHERE précoce avant GROUP BY
- ✅ require_partition_filter = true sur toutes les tables partitionnées
- ✅ Window functions plutôt que Self-JOINs
- ✅ APPROX_COUNT_DISTINCT pour les gros volumes

### Ta stack Data Warehouse

```
Fichiers Parquet dans GCS Bucket
    ↓
External Table (pointer vers GCS)
    ↓
Regular Table (données dans BigQuery)
    ↓
Table Partitionnée (découpage par date)
    ↓
Table Partitionnée + Clusterisée (optimisation maximale)
    ↓
Analytics / BI Tools
```

---

*Notes du Module 3 - DataTalks Club DE Zoomcamp*  
*Basées sur les expériences pratiques avec les Yellow Taxi Trip Records 2024*